In [1]:
# !pip install desilofhe-cu124 -q
# !pip install transformers==4.56.0 -q
# !pip install huggingface_hub==0.34.4 -q

In [ ]:
import os
import dotenv
dotenv.load_dotenv()

from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'))

/home/gshin22/miniconda3/envs/p10t28/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# EDIT HERE — all other cells depend on these values
# ═══════════════════════════════════════════════════════════════════════════

OPTION     = 'C'             # 'A' (PyTorch) | 'B' (N-packing FHE)
                             # 'C' (D-packing fused FHE) | 'D' (D-packing unfused FHE)
                             # 'E' (D-packing fused FHE + KV-cache generation)

INPUT_TEXT = '417+384=801'  # Input for options A, B, C, D; also used for variance profiling
DEPTH      = 6              # Run only the first DEPTH transformer blocks (1 to 6)
BASE_DIR   = '/data2/users/gshin22/fhe_output' # Root dir: BASE_DIR/checkpoints/, BASE_DIR/results/

# Option E only: token IDs for KV-cache generation
ALL_TOKENS  = [95, 4, 1, 7, 20, 3, 8, 4, 28, 8, 0, 1]  # '417+384=801'
PREFILL_LEN = 9   # first 9 tokens → prefill ('417+384='), rest → generation ('801')

# ═══════════════════════════════════════════════════════════════════════════
assert OPTION in ('A', 'B', 'C', 'D', 'E'), f'Unknown OPTION: {OPTION!r}'
assert 1 <= DEPTH <= 6, f'DEPTH must be 1..6, got {DEPTH}'
assert 1 <= PREFILL_LEN < len(ALL_TOKENS), 'PREFILL_LEN must be < len(ALL_TOKENS)'

print(f'OPTION  = {OPTION!r}')
print(f'INPUT   = {INPUT_TEXT!r}')
print(f'DEPTH   = {DEPTH}')
print(f'BASE_DIR= {BASE_DIR!r}')
if OPTION == 'E':
    print(f'TOKENS  = {ALL_TOKENS}')
    print(f'PREFILL = {PREFILL_LEN}, GEN = {len(ALL_TOKENS) - PREFILL_LEN}')

OPTION  = 'C'
INPUT   = '417+384=801'
DEPTH   = 6
BASE_DIR= '/data2/users/gshin22/fhe_output'


In [4]:
import sys
import json
import time
import pathlib
import datetime
import numpy as np
import torch
from collections import deque
from transformers import AutoModel, AutoTokenizer

sys.path.insert(0, os.path.abspath('.'))

from src.config import FHEConfig
from src.weights import (
    extract_weights_from_hf_model,
    profile_layernorm_variances,
    precompute_fused_weights,
    precompute_all_blocks,
)
from src.layernorm import apply_layernorm_rows
from src.muse_mixing import fhe_fused_mixing_layer, precompute_mixing_P_matrices
from src.muse_polymlp import fhe_fused_polymlp
from src.muse_block_unfused import fhe_unfused_mixing_layer, fhe_unfused_polymlp

if OPTION == 'B':
    from src.engine_n import (
        create_fhe_engine_n, create_keys_n, next_power_of_2,
        encrypt_matrix_n, decrypt_matrix_n,
    )
    from src.layernorm_n import layernorm_n
    from src.muse_mixing_n import fhe_mixing_layer_n
    from src.muse_polymlp_n import fhe_polymlp_n
else:
    from src.engine import create_fhe_engine, create_keys, bootstrap_if_needed, bs_kwargs

print('Imports OK')

/home/gshin22/miniconda3/envs/p10t28/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Imports OK


In [5]:
config = FHEConfig(
    embedding_dim=512,
    vocab_size=101,
    num_blocks=6,
    num_heads=8,
    max_mixing_length=64,
    hiddenmult=1,
    num_chunks=8,
    bootstrap_key_size='medium',
    bootstrap_stage_count=3,
    use_14_levels=False,
    invsqrt_iter=0,
    var_e=1e-5,
    min_var=0.5,
    max_var=5.0,
)
D  = config.embedding_dim   # 512
H  = config.num_heads        # 8
m  = config.max_mixing_length # 64
hs = config.head_size        # 64
print(config)

FHEConfig(
  embedding_dim=512, vocab_size=101,
  num_blocks=6, num_heads=8,
  max_mixing_length=64, hiddenmult=1,
  head_size=64, hidden_features=512,
  bottleneck_features=64,
  num_chunks=8, bootstrap_key_size='medium',
  use_small_bootstrap_key=False,
  bootstrap_stage_count=3,
  invsqrt_iter=0, var_e=1e-05,
  min_var=0.5, max_var=5.0
)


In [6]:
MODEL_NAME = 'giwone1330/3a3_m_tiny'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model      = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()
print(f'Model loaded. Params: {sum(p.numel() for p in model.parameters()):,}')

A new version of the following files was downloaded from https://huggingface.co/giwone1330/3a3_m_tiny:
- configuration_gpt2_muse.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


A new version of the following files was downloaded from https://huggingface.co/giwone1330/3a3_m_tiny:
- modeling_gpt2_muse.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Model loaded. Params: 8,846,848


In [7]:
# ─── PyTorch reference forward (always runs) ────────────────────────────────
# Captures per-component hidden states for all DEPTH blocks.
# pt_refs[l] = {'ln1', 'mixing', 'ln2', 'polymlp', 'block'} each (T, D) ndarray.

_tok_ref   = tokenizer(INPUT_TEXT, return_tensors='pt')
_ids_ref   = _tok_ref['input_ids']
T          = _ids_ref.shape[1]
_dev       = next(model.parameters()).device

with torch.no_grad():
    _wte_ref = model.transformer.wte(_ids_ref.to(_dev))
    _wpe_ref = model.transformer.wpe(torch.arange(T, device=_dev).unsqueeze(0))
    _hid_ref = _wte_ref + _wpe_ref

pt_refs = {}
for _l in range(DEPTH):
    _blk = model.transformer.h[_l]
    with torch.no_grad():
        _ln1  = _blk.ln_1(_hid_ref)
        _attn = _blk.attn(_ln1)
        _mix  = _attn[0] if isinstance(_attn, tuple) else _attn
        _res1 = _hid_ref + _mix
        _ln2  = _blk.ln_2(_res1)
        _mlp  = _blk.mlp(_ln2)
        _mlp  = _mlp[0] if isinstance(_mlp, tuple) else _mlp
        _hid_ref = _res1 + _mlp
    pt_refs[_l] = {
        'ln1':     _ln1[0].cpu().numpy(),
        'mixing':  _mix[0].cpu().numpy(),
        'ln2':     _ln2[0].cpu().numpy(),
        'polymlp': _mlp[0].cpu().numpy(),
        'block':   _hid_ref[0].cpu().numpy(),
    }
    print(f'  Block {_l}: block_out[0,:3]={_hid_ref[0,0,:3].cpu().numpy().round(4)}')

print(f'\nPyTorch reference complete. T={T}, DEPTH={DEPTH}')

  Block 0: block_out[0,:3]=[ 305.2745 -301.1354  224.8154]
  Block 1: block_out[0,:3]=[ 582.8955 -598.4778  239.5357]
  Block 2: block_out[0,:3]=[ 800.1822 -736.1198  194.2062]
  Block 3: block_out[0,:3]=[ 843.4708 -880.6902   58.8792]
  Block 4: block_out[0,:3]=[  901.1036 -1119.2423   184.0443]
  Block 5: block_out[0,:3]=[ 1116.9714 -1509.6536   -43.4983]

PyTorch reference complete. T=12, DEPTH=6


In [8]:
print('Extracting weights...')
weights          = extract_weights_from_hf_model(model, config)
all_block_weights = weights['blocks']
wte_w            = weights['wte_weight']   # (vocab, D)
wpe_w            = weights['wpe_weight']   # (max_pos, D)

print('Profiling LayerNorm variances...')
var_profile = profile_layernorm_variances(model, tokenizer, INPUT_TEXT,
                                          margin_low=0.5, margin_high=2.0)

def get_ln_bounds(block_idx, ln_idx):
    """Return {'min_var', 'max_var'} for block_idx, ln_idx (0=LN1, 1=LN2)."""
    entry = var_profile['per_ln'][block_idx * 2 + ln_idx]
    return {'min_var': entry['min_var'], 'max_var': entry['max_var']}

print(f'\nPer-LN bounds for first {DEPTH} block(s):')
for _l in range(DEPTH):
    b1 = get_ln_bounds(_l, 0); b2 = get_ln_bounds(_l, 1)
    print(f'  Block {_l}  LN1: [{b1["min_var"]:.3e}, {b1["max_var"]:.3e}]  '
          f'LN2: [{b2["min_var"]:.3e}, {b2["max_var"]:.3e}]')

# Fused weight precomputation (options C and E only)
all_fused = None
if OPTION in ('C', 'E'):
    _T_fw = T if OPTION == 'C' else len(ALL_TOKENS)
    print(f'\nPrecomputing fused weights for T={_T_fw}...')
    _t0 = time.time()
    all_fused = [precompute_fused_weights(all_block_weights[l], config, _T_fw)
                 for l in range(DEPTH)]
    print(f'Done in {time.time() - _t0:.1f}s')

Extracting weights...
  Block 0: c_attn_A=(512, 512), c_attn_C=(512, 512), c_fc=(512, 512), c_fc1=(64, 512), c_fc2=(512, 64), alpha=1.0000
  Block 1: c_attn_A=(512, 512), c_attn_C=(512, 512), c_fc=(512, 512), c_fc1=(64, 512), c_fc2=(512, 64), alpha=1.0000
  Block 2: c_attn_A=(512, 512), c_attn_C=(512, 512), c_fc=(512, 512), c_fc1=(64, 512), c_fc2=(512, 64), alpha=1.0000
  Block 3: c_attn_A=(512, 512), c_attn_C=(512, 512), c_fc=(512, 512), c_fc1=(64, 512), c_fc2=(512, 64), alpha=1.0000
  Block 4: c_attn_A=(512, 512), c_attn_C=(512, 512), c_fc=(512, 512), c_fc1=(64, 512), c_fc2=(512, 64), alpha=1.0000
  Block 5: c_attn_A=(512, 512), c_attn_C=(512, 512), c_fc=(512, 512), c_fc1=(64, 512), c_fc2=(512, 64), alpha=1.0000
Profiling LayerNorm variances...

Per-LN bounds for first 6 block(s):
  Block 0  LN1: [2.971e-03, 1.556e-01]  LN2: [3.812e+03, 5.080e+05]
  Block 1  LN1: [6.446e+03, 6.316e+05]  LN2: [1.010e+04, 6.409e+05]
  Block 2  LN1: [1.143e+04, 6.625e+05]  LN2: [1.923e+04, 7.880e+05]
  

In [9]:
if OPTION != 'A':
    if OPTION == 'B':
        _tok_n = tokenizer(INPUT_TEXT, return_tensors='pt')
        _T_n   = _tok_n['input_ids'].shape[1]
        engine, slot_count = create_fhe_engine_n(_T_n, config, track=True)
        print(f'N-packing engine: slot_count={slot_count}, max_level={engine.max_level}')
        secret_key = engine.create_secret_key()
        keys = create_keys_n(engine, secret_key, config)
        keys['secret_key'] = secret_key
    else:
        engine = create_fhe_engine(config, track=True)
        slot_count = engine.slot_count
        print(f'D-packing engine: slot_count={slot_count}, max_level={engine.max_level}')
        secret_key = engine.create_secret_key()
        keys = create_keys(engine, secret_key, config)
    print(f'Keys generated. Type: {"N-packing" if OPTION == "B" else "D-packing"}')

D-packing engine: slot_count=512, max_level=26


Keys generated. Type: D-packing


In [10]:
# ─── Metric snapshot utilities ──────────────────────────────────────────────
# Maps our 8 mutually exclusive categories to op_counts keys.
_OP_TYPE_TO_COUNT_KEYS = {
    'encrypt':      ['encrypt'],
    'decrypt':      ['decrypt'],
    'pt_ct_mult':   ['multiply_ct_pt'],
    'ct_ct_mult':   ['multiply_ct_ct', 'square'],
    'matrix_mult':  ['multiply_matrix'],
    'rotation':     ['rotate'],
    'add_subtract': ['add', 'subtract'],
    'bootstrap':    ['bootstrap', 'intt'],
}

def snap(eng):
    """Snapshot current engine counter state."""
    return {
        'counts':          dict(eng.op_counts),
        'times':           dict(getattr(eng, 'times_by_type', {})),
        'levels_consumed': dict(getattr(eng, 'levels_consumed_by_type', {})),
        'levels_restored': dict(getattr(eng, 'levels_restored_by_type', {})),
    }

def snap_delta(before, after):
    """
    Compute per-op-type metrics between two snapshots.
    Returns dict: {op_type: {'time': float, 'count': int, 'level': int}}
    'level' is negative for consumed levels, positive for restored.
    """
    result = {}
    for op, ck in _OP_TYPE_TO_COUNT_KEYS.items():
        count = sum(after['counts'].get(k, 0) - before['counts'].get(k, 0) for k in ck)
        if count == 0:
            continue
        t_delta  = after['times'].get(op, 0.0)           - before['times'].get(op, 0.0)
        lc_delta = after['levels_consumed'].get(op, 0)   - before['levels_consumed'].get(op, 0)
        lr_delta = after['levels_restored'].get(op, 0)   - before['levels_restored'].get(op, 0)
        result[op] = {
            'time':  round(t_delta, 4),
            'count': count,
            'level': round(-lc_delta + lr_delta, 0),  # negative=consumed, positive=restored
        }
    return result

print('Metric utilities defined (snap, snap_delta).')
print('8 operation categories:', list(_OP_TYPE_TO_COUNT_KEYS.keys()))

Metric utilities defined (snap, snap_delta).
8 operation categories: ['encrypt', 'decrypt', 'pt_ct_mult', 'ct_ct_mult', 'matrix_mult', 'rotation', 'add_subtract', 'bootstrap']


In [11]:
# ─── Checkpoint utilities ────────────────────────────────────────────────────
# Uses engine.write_ciphertext / engine.read_ciphertext (desilofhe native API).
# Directory structure:
#   BASE_DIR/checkpoints/{option}/block_{l}/   ← enc_hidden + meta.json + block_stats.json
#   BASE_DIR/checkpoints/E/prefill_{l}_cache/  ← cache_Ch for option E prefill
#   BASE_DIR/checkpoints/E/gen_step_{j}_cache_{l}/ ← cache_Ch for option E generation

def ckpt_dir_for(option, block_or_key, suffix=''):
    """Return Path to the checkpoint directory for option/block/suffix."""
    p = pathlib.Path(BASE_DIR) / 'checkpoints' / str(option) / f'block_{block_or_key}{suffix}'
    return p

def ckpt_exists(option, block_or_key, suffix=''):
    """True if block_stats.json exists in the checkpoint directory (block fully done)."""
    return (ckpt_dir_for(option, block_or_key, suffix) / 'block_stats.json').exists()

def save_ct_list(eng, ct_list, ckpt_dir):
    """
    Save a list of Ciphertext objects using engine.write_ciphertext.
    Writes {0.ct, 1.ct, ...} + meta.json with count.
    """
    d = pathlib.Path(ckpt_dir)
    d.mkdir(parents=True, exist_ok=True)
    for i, ct in enumerate(ct_list):
        eng.write_ciphertext(ct, str(d / f'{i}.ct'))
    with open(d / 'meta.json', 'w') as _f:
        json.dump({'count': len(ct_list)}, _f)
    print(f'  [CKPT] Saved {len(ct_list)} ct(s) → {d}')

def load_ct_list(eng, ckpt_dir):
    """Load Ciphertext list from ckpt_dir. Returns List[Ciphertext]."""
    d = pathlib.Path(ckpt_dir)
    with open(d / 'meta.json') as _f:
        _meta = json.load(_f)
    ct_list = [eng.read_ciphertext(str(d / f'{i}.ct')) for i in range(_meta['count'])]
    print(f'  [CKPT] Loaded {len(ct_list)} ct(s) from {d}')
    return ct_list

def save_cache_Ch(eng, cache_Ch, ckpt_dir):
    """
    Save a List[H] of deques of Ciphertext (KV cache for one block).
    Writes h{h}_t{k}.ct files + meta.json with num_heads and window_sizes.
    """
    d = pathlib.Path(ckpt_dir)
    d.mkdir(parents=True, exist_ok=True)
    meta = {'num_heads': len(cache_Ch), 'window_sizes': []}
    for h, dq in enumerate(cache_Ch):
        items = list(dq)
        meta['window_sizes'].append(len(items))
        for k, ct in enumerate(items):
            eng.write_ciphertext(ct, str(d / f'h{h}_t{k}.ct'))
    with open(d / 'meta.json', 'w') as _f:
        json.dump(meta, _f)
    print(f'  [CKPT] Saved cache_Ch ({len(cache_Ch)} heads) → {d}')

def load_cache_Ch(eng, ckpt_dir, maxlen):
    """
    Load cache_Ch from ckpt_dir.
    Returns List[H] of deques(maxlen=maxlen) of Ciphertext.
    """
    d = pathlib.Path(ckpt_dir)
    with open(d / 'meta.json') as _f:
        meta = json.load(_f)
    cache_Ch = []
    for h in range(meta['num_heads']):
        dq = deque(maxlen=maxlen)
        for k in range(meta['window_sizes'][h]):
            dq.append(eng.read_ciphertext(str(d / f'h{h}_t{k}.ct')))
        cache_Ch.append(dq)
    print(f'  [CKPT] Loaded cache_Ch ({meta["num_heads"]} heads) from {d}')
    return cache_Ch

def result_path(option):
    """Return path for the JSON result file with timestamp."""
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    p  = pathlib.Path(BASE_DIR) / 'results'
    p.mkdir(parents=True, exist_ok=True)
    return p / f'{option}_{ts}.json'

print('Checkpoint utilities defined.')
print(f'Checkpoint root: {pathlib.Path(BASE_DIR).absolute() / "checkpoints"}')

Checkpoint utilities defined.
Checkpoint root: /data2/users/gshin22/fhe_output/checkpoints


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION A — PyTorch inference
# ═══════════════════════════════════════════════════════════════════════════
if OPTION == 'A':
    _tok_a  = tokenizer(INPUT_TEXT, return_tensors='pt')
    _ids_a  = _tok_a['input_ids']
    _T_a    = _ids_a.shape[1]
    _dev_a  = next(model.parameters()).device

    with torch.no_grad():
        _wte_a   = model.transformer.wte(_ids_a.to(_dev_a))
        _wpe_a   = model.transformer.wpe(torch.arange(_T_a, device=_dev_a).unsqueeze(0))
        _hidden_a = _wte_a + _wpe_a

    results = {
        'option':       'A',
        'input_text':   INPUT_TEXT,
        'depth':        DEPTH,
        'blocks':       {},
        'total_time_s': 0.0,
    }

    _t0_total_a = time.time()

    for _l in range(DEPTH):
        _blk_a     = model.transformer.h[_l]
        _block_data = {}
        _t_block_a = time.time()

        with torch.no_grad():
            # LN1
            _t0 = time.time()
            _ln1_a = _blk_a.ln_1(_hidden_a)
            _block_data['ln1'] = {
                'time_s': time.time() - _t0,
                'hidden': _ln1_a[0].cpu().numpy().tolist(),
            }

            # MixingLayer
            _t0 = time.time()
            _attn_a = _blk_a.attn(_ln1_a)
            _mix_a  = _attn_a[0] if isinstance(_attn_a, tuple) else _attn_a
            _block_data['mixing'] = {
                'time_s': time.time() - _t0,
                'hidden': _mix_a[0].cpu().numpy().tolist(),
            }

            _res1_a = _hidden_a + _mix_a

            # LN2
            _t0 = time.time()
            _ln2_a = _blk_a.ln_2(_res1_a)
            _block_data['ln2'] = {
                'time_s': time.time() - _t0,
                'hidden': _ln2_a[0].cpu().numpy().tolist(),
            }

            # PolyMlp
            _t0 = time.time()
            _mlp_a = _blk_a.mlp(_ln2_a)
            _mlp_a = _mlp_a[0] if isinstance(_mlp_a, tuple) else _mlp_a
            _block_data['polymlp'] = {
                'time_s': time.time() - _t0,
                'hidden': _mlp_a[0].cpu().numpy().tolist(),
            }

            _hidden_a = _res1_a + _mlp_a
            _block_data['block'] = {
                'time_s': time.time() - _t_block_a,
                'hidden': _hidden_a[0].cpu().numpy().tolist(),
            }

        results['blocks'][f'block{_l}'] = _block_data
        print(f'Block {_l} done in {_block_data["block"]["time_s"]:.4f}s')

    results['total_time_s'] = time.time() - _t0_total_a
    print(f'\nOption A complete in {results["total_time_s"]:.4f}s')

In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION B — N-packing FHE inference
# ═══════════════════════════════════════════════════════════════════════════
if OPTION == 'B':
    _tok_b  = tokenizer(INPUT_TEXT, return_tensors='pt')
    _ids_b  = _tok_b['input_ids']
    _T_b    = _ids_b.shape[1]
    _dev_b  = next(model.parameters()).device

    with torch.no_grad():
        _wte_b = model.transformer.wte(_ids_b.to(_dev_b))
        _wpe_b = model.transformer.wpe(torch.arange(_T_b, device=_dev_b).unsqueeze(0))
        _X_emb_b = (_wte_b + _wpe_b)[0].cpu().numpy()  # (T, D)

    # Encrypt (N-packing: D column-ciphertexts of T slots each)
    print(f'Encrypting {D} column-ciphertexts (T={_T_b}, slot_count={slot_count})...')
    enc_cols_b = encrypt_matrix_n(engine, secret_key, _X_emb_b, _T_b, D, slot_count)

    def _dec_n(enc_cols, n_rows):
        return decrypt_matrix_n(engine, secret_key, enc_cols, n_rows, D)

    results = {
        'option':       'B',
        'input_text':   INPUT_TEXT,
        'depth':        DEPTH,
        'blocks':       {},
    }
    _t0_total_b = time.time()

    for _l in range(DEPTH):
        _bw_b = all_block_weights[_l]
        _b1   = get_ln_bounds(_l, 0)
        _b2   = get_ln_bounds(_l, 1)
        _ckpt = ckpt_dir_for('B', _l)

        if ckpt_exists('B', _l):
            print(f'[CKPT] Block {_l}: loading from checkpoint...')
            enc_cols_b = load_ct_list(engine, _ckpt)
            with open(_ckpt / 'block_stats.json') as _f:
                results['blocks'][f'block{_l}'] = json.load(_f)
            continue

        engine.reset_counters()
        _block_stats = {}
        _t_block     = time.time()

        # LN1
        _s0 = snap(engine); _t0 = time.time()
        _enc_ln1_b = layernorm_n(engine, keys, config, enc_cols_b, D, _T_b,
                                 gamma=_bw_b['ln_1_weight'], beta=_bw_b['ln_1_bias'],
                                 min_var=_b1['min_var'], max_var=_b1['max_var'])
        _block_stats['ln1'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_n(_enc_ln1_b, _T_b).tolist(),
        }

        # MixingLayer
        _s0 = snap(engine); _t0 = time.time()
        _enc_attn_b = fhe_mixing_layer_n(engine, keys, config, _enc_ln1_b, _bw_b, _T_b, slot_count)
        _block_stats['mixing'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_n(_enc_attn_b, _T_b).tolist(),
        }

        # Residual 1
        enc_cols_b = [engine.add(enc_cols_b[_d], _enc_attn_b[_d]) for _d in range(D)]

        # LN2
        _s0 = snap(engine); _t0 = time.time()
        _enc_ln2_b = layernorm_n(engine, keys, config, enc_cols_b, D, _T_b,
                                 gamma=_bw_b['ln_2_weight'], beta=_bw_b['ln_2_bias'],
                                 min_var=_b2['min_var'], max_var=_b2['max_var'])
        _block_stats['ln2'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_n(_enc_ln2_b, _T_b).tolist(),
        }

        # PolyMlp
        _s0 = snap(engine); _t0 = time.time()
        _enc_mlp_b = fhe_polymlp_n(engine, keys, config, _enc_ln2_b, _bw_b, _T_b, slot_count)
        _block_stats['polymlp'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_n(_enc_mlp_b, _T_b).tolist(),
        }

        # Residual 2
        enc_cols_b = [engine.add(enc_cols_b[_d], _enc_mlp_b[_d]) for _d in range(D)]
        _block_stats['block'] = {
            'time_s': time.time() - _t_block,
            'hidden': _dec_n(enc_cols_b, _T_b).tolist(),
        }

        results['blocks'][f'block{_l}'] = _block_stats

        # Save checkpoint
        save_ct_list(engine, enc_cols_b, _ckpt)
        with open(_ckpt / 'block_stats.json', 'w') as _f:
            json.dump(_block_stats, _f)

        print(f'Block {_l} done in {_block_stats["block"]["time_s"]:.1f}s')

    results['total_time_s'] = time.time() - _t0_total_b
    print(f'\nOption B complete. Total: {results["total_time_s"]:.1f}s')

In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION C — D-packing fused FHE inference
# OPTION D — D-packing unfused FHE inference
# ═══════════════════════════════════════════════════════════════════════════
if OPTION in ('C', 'D'):
    _tok_cd = tokenizer(INPUT_TEXT, return_tensors='pt')
    _ids_cd = _tok_cd['input_ids']
    _T_cd   = _ids_cd.shape[1]
    _dev_cd = next(model.parameters()).device

    with torch.no_grad():
        _wte_cd = model.transformer.wte(_ids_cd.to(_dev_cd))
        _wpe_cd = model.transformer.wpe(torch.arange(_T_cd, device=_dev_cd).unsqueeze(0))
        _X_emb_cd = (_wte_cd + _wpe_cd)[0].cpu().numpy()  # (T, D)

    # Encrypt (D-packing: T row-ciphertexts of D slots each)
    print(f'Encrypting {_T_cd} row-ciphertexts (slot_count={slot_count})...')
    enc_hidden_cd = [engine.encrypt(_X_emb_cd[_t].tolist(), secret_key) for _t in range(_T_cd)]

    def _dec_d(enc_rows):
        """Decrypt T D-packing row-ciphertexts → (T, D) numpy."""
        _out = []
        for _ct in enc_rows:
            _vals = engine.decrypt(_ct, secret_key)
            _out.append(np.array(_vals)[:D])
        return np.stack(_out)

    results = {
        'option':     OPTION,
        'input_text': INPUT_TEXT,
        'depth':      DEPTH,
        'blocks':     {},
    }
    _t0_total_cd = time.time()

    for _l in range(DEPTH):
        _bw_cd = all_block_weights[_l]
        _fw_cd = all_fused[_l] if OPTION == 'C' else None
        _b1    = get_ln_bounds(_l, 0)
        _b2    = get_ln_bounds(_l, 1)
        _ckpt  = ckpt_dir_for(OPTION, _l)

        if ckpt_exists(OPTION, _l):
            print(f'[CKPT] Block {_l}: loading from checkpoint...')
            enc_hidden_cd = load_ct_list(engine, _ckpt)
            with open(_ckpt / 'block_stats.json') as _f:
                results['blocks'][f'block{_l}'] = json.load(_f)
            continue

        engine.reset_counters()
        _block_stats = {}
        _t_block     = time.time()

        # LN1
        _s0 = snap(engine); _t0 = time.time()
        _enc_ln1_cd = apply_layernorm_rows(
            engine, keys, config, enc_hidden_cd, _T_cd,
            _bw_cd['ln_1_weight'], _bw_cd['ln_1_bias'],
            min_var=_b1['min_var'], max_var=_b1['max_var'],
        )
        _block_stats['ln1'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_d(_enc_ln1_cd).tolist(),
        }

        # MixingLayer (fused or unfused)
        _s0 = snap(engine); _t0 = time.time()
        if OPTION == 'C':
            _enc_attn_cd = fhe_fused_mixing_layer(
                engine, keys, config, _enc_ln1_cd, _fw_cd, _T_cd)
        else:
            _enc_attn_cd = fhe_unfused_mixing_layer(
                engine, keys, config, _enc_ln1_cd, _bw_cd, _T_cd)
        _block_stats['mixing'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_d(_enc_attn_cd).tolist(),
        }

        # Residual 1
        enc_hidden_cd = [engine.add(enc_hidden_cd[_t], _enc_attn_cd[_t]) for _t in range(_T_cd)]

        # LN2
        _s0 = snap(engine); _t0 = time.time()
        _enc_ln2_cd = apply_layernorm_rows(
            engine, keys, config, enc_hidden_cd, _T_cd,
            _bw_cd['ln_2_weight'], _bw_cd['ln_2_bias'],
            min_var=_b2['min_var'], max_var=_b2['max_var'],
        )
        _block_stats['ln2'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_d(_enc_ln2_cd).tolist(),
        }

        # PolyMlp (fused or unfused)
        _s0 = snap(engine); _t0 = time.time()
        if OPTION == 'C':
            _enc_mlp_cd = fhe_fused_polymlp(
                engine, keys, config, _enc_ln2_cd, _fw_cd, _T_cd)
        else:
            _enc_mlp_cd = fhe_unfused_polymlp(
                engine, keys, config, _enc_ln2_cd, _bw_cd, _T_cd)
        _block_stats['polymlp'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_d(_enc_mlp_cd).tolist(),
        }

        # Residual 2
        enc_hidden_cd = [engine.add(enc_hidden_cd[_t], _enc_mlp_cd[_t]) for _t in range(_T_cd)]
        _block_stats['block'] = {
            'time_s': time.time() - _t_block,
            'hidden': _dec_d(enc_hidden_cd).tolist(),
        }

        results['blocks'][f'block{_l}'] = _block_stats

        # Save checkpoint
        save_ct_list(engine, enc_hidden_cd, _ckpt)
        with open(_ckpt / 'block_stats.json', 'w') as _f:
            json.dump(_block_stats, _f)

        print(f'Block {_l} done in {_block_stats["block"]["time_s"]:.1f}s')
        engine.print_summary(f'Block {_l} op summary')

    results['total_time_s'] = time.time() - _t0_total_cd
    print(f'\nOption {OPTION} complete. Total: {results["total_time_s"]:.1f}s')

Encrypting 12 row-ciphertexts (slot_count=512)...


      [FusedMix] Head 0/8 done in 47.91s


      [FusedMix] Head 1/8 done in 48.18s


      [FusedMix] Head 2/8 done in 48.32s


      [FusedMix] Head 3/8 done in 48.22s


...

      [FusedMix] Head 4/8 done in 81.79s


      [FusedMix] Head 5/8 done in 82.13s


      [FusedMix] Head 6/8 done in 81.98s


      [FusedMix] Head 7/8 done in 82.01s


  [CKPT] Saved 12 ct(s) → /data2/users/gshin22/fhe_output/checkpoints/C/block_5
Block 5 done in 1051.4s

  --- Block 5 op summary ---
          multiply_ct_ct: 1548
          multiply_ct_pt: 2436
         multiply_matrix: 240
                     add: 9432
                subtract: 408
                  rotate: 8964
                  square: 312
               bootstrap: 456
                 encrypt: 24
                 decrypt: 60
                   clone: 24
                    intt: 456
         levels_consumed: 5280
         levels_restored: 3024
         net_levels_used: 2256
        total_bootstraps: 456
       lv5→10 (×96)
      fused_mix_head_7 lv5→10 (×96)
      fused_mix_head_7 lv2→10 (×36)
      fused_mix_head_0 lv2→10 (×24)
      fused_mix_head_1 lv2→10 (×24)
      fused_mix_head_2 lv2→10 (×24)
      fused_mix_head_3 lv2→10 (×24)
      fused_mix_head_4 lv2→10 (×24)
      fused_mix_head_5 lv2→10 (×24)
      fused_mix_head_6 lv2→10 (×24)
       lv4→10 (×12)
       lv2→10 (×12

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION E — KV-cache function definitions
# ═══════════════════════════════════════════════════════════════════════════
if OPTION == 'E':
    # ─── Encrypt / Decrypt helpers ─────────────────────────────────────────
    def _enc_row_e(tid, pos):
        row    = wte_w[tid] + wpe_w[pos]
        padded = np.zeros(slot_count, dtype=np.float64)
        padded[:D] = row
        return engine.encrypt(padded.tolist(), secret_key)

    def _enc_rows_e(token_ids, start_pos):
        return [_enc_row_e(tid, start_pos + i) for i, tid in enumerate(token_ids)]

    def _dec_row_e(ct):
        return np.array(engine.decrypt(ct, secret_key))[:D]

    def _dec_rows_e(enc_rows):
        return np.stack([_dec_row_e(ct) for ct in enc_rows])

    def _fhe_ln_e(enc_rows, T_ln, gamma, beta, min_var, max_var):
        return apply_layernorm_rows(
            engine, keys, config, enc_rows, T_ln, gamma, beta,
            min_var=min_var, max_var=max_var,
        )

    def _fhe_polymlp_e(enc_rows, fw, T_poly):
        return fhe_fused_polymlp(engine, keys, config, enc_rows, fw, T_poly)

    # ─── KV-Cached Mixing Layer — Prefill ──────────────────────────────────
    def _fhe_mix_prefill_e(enc_ln1_rows, bw, T_pre):
        """
        Prefill mixing layer: computes B_h for all T_pre tokens and saves
        C_h ciphertexts into the KV cache (one deque per head).
        Returns (enc_out, cache_Ch).
        """
        rk = keys['rotation_key']
        rl = keys['relin_key']
        W_A = bw['c_attn_A_weight']     # (H*m, D)
        W_C = bw['c_attn_C_weight']     # (D, D)
        W_P = bw['c_proj_attn_weight']  # (D, D)
        cache_Ch   = [deque(maxlen=m) for _ in range(H)]
        enc_B_rows = [None] * T_pre

        for h in range(H):
            _t_h = time.time()
            W_A_h = W_A[h * m:(h + 1) * m, :]   # (m, D)
            W_C_h = W_C[h * hs:(h + 1) * hs, :] # (hs, D)
            P_mats = precompute_mixing_P_matrices(W_A_h, T_pre, m, D)

            enc_Dh = []
            for i in range(T_pre):
                ct = bootstrap_if_needed(engine, enc_ln1_rows[i], **bs_kwargs(keys))
                padded_P = np.zeros((slot_count, slot_count), dtype=np.float64)
                padded_P[:T_pre, :D] = P_mats[i]
                enc_Dh.append(engine.multiply_matrix(padded_P, ct, rk))

            padded_C = np.zeros((slot_count, slot_count), dtype=np.float64)
            padded_C[:hs, :D] = W_C_h
            c_bias_pad = None
            if bw['c_attn_C_bias'] is not None:
                c_bias_pad = np.zeros(slot_count, dtype=np.float64)
                c_bias_pad[:hs] = bw['c_attn_C_bias'][h * hs:(h + 1) * hs]

            enc_Ch = []
            for t in range(T_pre):
                ct = bootstrap_if_needed(engine, enc_ln1_rows[t], **bs_kwargs(keys))
                c_ct = engine.multiply_matrix(padded_C, ct, rk)
                if c_bias_pad is not None:
                    c_ct = engine.add(c_ct, c_bias_pad.tolist())
                enc_Ch.append(c_ct)
                cache_Ch[h].append(c_ct)

            enc_Dh    = [bootstrap_if_needed(engine, ct, **bs_kwargs(keys)) for ct in enc_Dh]
            enc_Ch_bs = [bootstrap_if_needed(engine, ct, **bs_kwargs(keys)) for ct in enc_Ch]

            for i in range(T_pre):
                B_h_i = None
                for j in range(T_pre):
                    mask_j    = np.zeros(slot_count, dtype=np.float64)
                    mask_j[j] = 1.0
                    d_ij = engine.multiply(enc_Dh[i], mask_j.tolist())
                    if j > 0:
                        d_ij = engine.rotate(d_ij, rk, delta=-j)
                    shift = 1
                    while shift < hs:
                        d_ij  = engine.add(d_ij, engine.rotate(d_ij, rk, delta=shift))
                        shift *= 2
                    prod  = engine.multiply(d_ij, enc_Ch_bs[j], rl)
                    B_h_i = prod if B_h_i is None else engine.add(B_h_i, prod)
                if h > 0:
                    B_h_i = engine.rotate(B_h_i, rk, delta=h * hs)
                enc_B_rows[i] = (B_h_i if enc_B_rows[i] is None
                                 else engine.add(enc_B_rows[i], B_h_i))
            print(f'    [PrefillMix] head {h}/{H}  {time.time()-_t_h:.1f}s', flush=True)

        padded_proj = np.zeros((slot_count, slot_count), dtype=np.float64)
        padded_proj[:D, :D] = W_P
        proj_bias = np.zeros(slot_count, dtype=np.float64)
        if bw['c_proj_attn_bias'] is not None:
            proj_bias[:D] = bw['c_proj_attn_bias']

        enc_out = []
        for t in range(T_pre):
            ct = bootstrap_if_needed(engine, enc_B_rows[t], **bs_kwargs(keys))
            r  = engine.multiply_matrix(padded_proj, ct, rk)
            enc_out.append(engine.add(r, proj_bias.tolist()))

        return enc_out, cache_Ch

    # ─── KV-Cached Mixing Layer — Generation Step ───────────────────────────
    def _fhe_mix_step_e(enc_ln1_new, bw, cache_Ch, t_pos):
        """
        Single-token mixing using the KV cache.
        Pushes new C_h into cache_Ch[h] in-place.
        Returns enc_out Ciphertext for the new token.
        """
        rk = keys['rotation_key']
        rl = keys['relin_key']
        W_A = bw['c_attn_A_weight']
        W_C = bw['c_attn_C_weight']
        W_P = bw['c_proj_attn_weight']
        enc_B_out = None

        for h in range(H):
            W_A_h  = W_A[h * m:(h + 1) * m, :]
            W_C_h  = W_C[h * hs:(h + 1) * hs, :]
            window = min(t_pos + 1, m)

            P_h_t = np.zeros((slot_count, slot_count), dtype=np.float64)
            for k in range(window):
                P_h_t[k, :D] = W_A_h[k, :] / (t_pos + 1)

            ct_ln1   = bootstrap_if_needed(engine, enc_ln1_new, **bs_kwargs(keys))
            enc_D_row = engine.multiply_matrix(P_h_t, ct_ln1, rk)

            padded_C = np.zeros((slot_count, slot_count), dtype=np.float64)
            padded_C[:hs, :D] = W_C_h
            ct_ln1_c = bootstrap_if_needed(engine, enc_ln1_new, **bs_kwargs(keys))
            enc_C_new = engine.multiply_matrix(padded_C, ct_ln1_c, rk)
            if bw['c_attn_C_bias'] is not None:
                cb = np.zeros(slot_count, dtype=np.float64)
                cb[:hs] = bw['c_attn_C_bias'][h * hs:(h + 1) * hs]
                enc_C_new = engine.add(enc_C_new, cb.tolist())
            cache_Ch[h].append(enc_C_new)

            window_cts = list(cache_Ch[h])
            actual_w   = len(window_cts)
            enc_D_bs   = bootstrap_if_needed(engine, enc_D_row, **bs_kwargs(keys))

            B_h = None
            for k in range(actual_w):
                mask_k    = np.zeros(slot_count, dtype=np.float64)
                mask_k[k] = 1.0
                d_k = engine.multiply(enc_D_bs, mask_k.tolist())
                if k > 0:
                    d_k = engine.rotate(d_k, rk, delta=-k)
                shift = 1
                while shift < hs:
                    d_k  = engine.add(d_k, engine.rotate(d_k, rk, delta=shift))
                    shift *= 2
                c_h_tk = bootstrap_if_needed(
                    engine, window_cts[actual_w - 1 - k], **bs_kwargs(keys))
                prod = engine.multiply(d_k, c_h_tk, rl)
                B_h  = prod if B_h is None else engine.add(B_h, prod)

            if h > 0:
                B_h = engine.rotate(B_h, rk, delta=h * hs)
            enc_B_out = B_h if enc_B_out is None else engine.add(enc_B_out, B_h)

        padded_proj = np.zeros((slot_count, slot_count), dtype=np.float64)
        padded_proj[:D, :D] = W_P
        proj_bias = np.zeros(slot_count, dtype=np.float64)
        if bw['c_proj_attn_bias'] is not None:
            proj_bias[:D] = bw['c_proj_attn_bias']

        enc_B_bs = bootstrap_if_needed(engine, enc_B_out, **bs_kwargs(keys))
        enc_out  = engine.multiply_matrix(padded_proj, enc_B_bs, rk)
        return engine.add(enc_out, proj_bias.tolist())

    print('Option E: KV-cache function definitions done.')

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION E — KV-cache prefill + generation execution
# ═══════════════════════════════════════════════════════════════════════════
if OPTION == 'E':
    _T_pre_e   = PREFILL_LEN
    _GEN_TOKS  = ALL_TOKENS[PREFILL_LEN:]
    _PRE_TOKS  = ALL_TOKENS[:PREFILL_LEN]
    _N_GEN     = len(_GEN_TOKS)

    results = {
        'option':      'E',
        'all_tokens':  ALL_TOKENS,
        'prefill_len': PREFILL_LEN,
        'depth':       DEPTH,
        'prefill':     {},
        'generation':  {},
    }

    # kv_caches[l] = cache_Ch for block l (List[H deques])
    kv_caches = [None] * DEPTH

    # ─── Prefill ────────────────────────────────────────────────────────────
    print('=' * 60)
    print(f'PREFILL: {_T_pre_e} tokens, {DEPTH} block(s)')
    print('=' * 60)

    enc_x_e = _enc_rows_e(_PRE_TOKS, 0)

    for _l in range(DEPTH):
        _bw_e  = all_block_weights[_l]
        _fw_e  = all_fused[_l]
        _b1    = get_ln_bounds(_l, 0)
        _b2    = get_ln_bounds(_l, 1)
        _ckpt_pre   = ckpt_dir_for('E', f'pre_{_l}')
        _ckpt_cache = ckpt_dir_for('E', f'pre_{_l}_cache')

        if ckpt_exists('E', f'pre_{_l}'):
            print(f'[CKPT] Prefill block {_l}: loading...')
            enc_x_e     = load_ct_list(engine, _ckpt_pre)
            kv_caches[_l] = load_cache_Ch(engine, _ckpt_cache, maxlen=m)
            with open(_ckpt_pre / 'block_stats.json') as _f:
                results['prefill'][f'block{_l}'] = json.load(_f)
            continue

        engine.reset_counters()
        _blk_stats = {}
        _t_block   = time.time()

        # LN1
        _s0 = snap(engine); _t0 = time.time()
        _enc_ln1_e = _fhe_ln_e(enc_x_e, _T_pre_e,
                                _bw_e['ln_1_weight'], _bw_e['ln_1_bias'],
                                _b1['min_var'], _b1['max_var'])
        _blk_stats['ln1'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_rows_e(_enc_ln1_e).tolist(),
        }

        # MixingLayer (prefill with cache save)
        _s0 = snap(engine); _t0 = time.time()
        _enc_attn_e, kv_caches[_l] = _fhe_mix_prefill_e(_enc_ln1_e, _bw_e, _T_pre_e)
        _blk_stats['mixing'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_rows_e(_enc_attn_e).tolist(),
        }

        # Residual 1
        enc_x_e = [engine.add(enc_x_e[_t], _enc_attn_e[_t]) for _t in range(_T_pre_e)]

        # LN2
        _s0 = snap(engine); _t0 = time.time()
        _enc_ln2_e = _fhe_ln_e(enc_x_e, _T_pre_e,
                                _bw_e['ln_2_weight'], _bw_e['ln_2_bias'],
                                _b2['min_var'], _b2['max_var'])
        _blk_stats['ln2'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_rows_e(_enc_ln2_e).tolist(),
        }

        # PolyMlp
        _s0 = snap(engine); _t0 = time.time()
        _enc_mlp_e = _fhe_polymlp_e(_enc_ln2_e, _fw_e, _T_pre_e)
        _blk_stats['polymlp'] = {
            'time_s':     time.time() - _t0,
            'operations': snap_delta(_s0, snap(engine)),
            'hidden':     _dec_rows_e(_enc_mlp_e).tolist(),
        }

        # Residual 2
        enc_x_e = [engine.add(enc_x_e[_t], _enc_mlp_e[_t]) for _t in range(_T_pre_e)]
        _blk_stats['block'] = {
            'time_s': time.time() - _t_block,
            'hidden': _dec_rows_e(enc_x_e).tolist(),
        }

        results['prefill'][f'block{_l}'] = _blk_stats

        # Save checkpoint: encrypted hidden + KV cache
        save_ct_list(engine, enc_x_e, _ckpt_pre)
        save_cache_Ch(engine, kv_caches[_l], _ckpt_cache)
        with open(_ckpt_pre / 'block_stats.json', 'w') as _f:
            json.dump(_blk_stats, _f)

        print(f'Prefill block {_l} done in {_blk_stats["block"]["time_s"]:.1f}s')

    # ─── Cached Generation ──────────────────────────────────────────────────
    print('\n' + '=' * 60)
    print(f'CACHED GENERATION: {_N_GEN} token(s), {DEPTH} block(s)')
    print('=' * 60)

    for _step, _tok_id in enumerate(_GEN_TOKS):
        _t_pos  = PREFILL_LEN + _step
        _gen_key = f'gen_{_step}'
        _ckpt_gen = ckpt_dir_for('E', _gen_key)

        print(f'\n── Step {_step + 1}/{_N_GEN}  (pos={_t_pos}, token={_tok_id}) ──')

        if ckpt_exists('E', _gen_key):
            print(f'[CKPT] Gen step {_step}: loading...')
            _loaded_enc = load_ct_list(engine, _ckpt_gen)
            enc_x_new_e = _loaded_enc[0]
            for _l in range(DEPTH):
                _ckpt_cg = ckpt_dir_for('E', f'{_gen_key}_cache_{_l}')
                kv_caches[_l] = load_cache_Ch(engine, _ckpt_cg, maxlen=m)
            with open(_ckpt_gen / 'block_stats.json') as _f:
                results['generation'][_gen_key] = json.load(_f)
            continue

        enc_x_new_e = _enc_row_e(_tok_id, _t_pos)
        _gen_stats  = {'token_id': _tok_id, 't_pos': _t_pos, 'blocks': {}}

        for _l in range(DEPTH):
            _bw_e = all_block_weights[_l]
            _fw_e = all_fused[_l]
            _b1   = get_ln_bounds(_l, 0)
            _b2   = get_ln_bounds(_l, 1)

            engine.reset_counters()
            _blk_stats = {}
            _t_block   = time.time()

            # LN1 (single token)
            _s0 = snap(engine); _t0 = time.time()
            _enc_ln1_list = _fhe_ln_e([enc_x_new_e], 1,
                                       _bw_e['ln_1_weight'], _bw_e['ln_1_bias'],
                                       _b1['min_var'], _b1['max_var'])
            _blk_stats['ln1'] = {
                'time_s':     time.time() - _t0,
                'operations': snap_delta(_s0, snap(engine)),
                'hidden':     [_dec_row_e(_enc_ln1_list[0]).tolist()],
            }

            # MixingLayer (cached step)
            _s0 = snap(engine); _t0 = time.time()
            _enc_attn_new = _fhe_mix_step_e(_enc_ln1_list[0], _bw_e, kv_caches[_l], _t_pos)
            _blk_stats['mixing'] = {
                'time_s':     time.time() - _t0,
                'operations': snap_delta(_s0, snap(engine)),
                'hidden':     [_dec_row_e(_enc_attn_new).tolist()],
            }

            enc_x_new_e = engine.add(enc_x_new_e, _enc_attn_new)

            # LN2
            _s0 = snap(engine); _t0 = time.time()
            _enc_ln2_list = _fhe_ln_e([enc_x_new_e], 1,
                                       _bw_e['ln_2_weight'], _bw_e['ln_2_bias'],
                                       _b2['min_var'], _b2['max_var'])
            _blk_stats['ln2'] = {
                'time_s':     time.time() - _t0,
                'operations': snap_delta(_s0, snap(engine)),
                'hidden':     [_dec_row_e(_enc_ln2_list[0]).tolist()],
            }

            # PolyMlp
            _s0 = snap(engine); _t0 = time.time()
            _enc_mlp_list = _fhe_polymlp_e(_enc_ln2_list, _fw_e, 1)
            _blk_stats['polymlp'] = {
                'time_s':     time.time() - _t0,
                'operations': snap_delta(_s0, snap(engine)),
                'hidden':     [_dec_row_e(_enc_mlp_list[0]).tolist()],
            }

            enc_x_new_e = engine.add(enc_x_new_e, _enc_mlp_list[0])
            _blk_stats['block'] = {
                'time_s': time.time() - _t_block,
                'hidden': [_dec_row_e(enc_x_new_e).tolist()],
            }

            _gen_stats['blocks'][f'block{_l}'] = _blk_stats

        results['generation'][_gen_key] = _gen_stats

        # Save checkpoint: final enc_x_new + all updated kv_caches
        save_ct_list(engine, [enc_x_new_e], _ckpt_gen)
        for _l in range(DEPTH):
            _ckpt_cg = ckpt_dir_for('E', f'{_gen_key}_cache_{_l}')
            save_cache_Ch(engine, kv_caches[_l], _ckpt_cg)
        with open(_ckpt_gen / 'block_stats.json', 'w') as _f:
            json.dump(_gen_stats, _f)

        print(f'Gen step {_step} done.')

    print('\nOption E complete.')

In [17]:
# ─── Save results to JSON ────────────────────────────────────────────────────
_out_path = result_path(OPTION)
with open(_out_path, 'w') as _f:
    json.dump(results, _f)

print(f'Results saved → {_out_path}')
print(f'File size: {_out_path.stat().st_size / 1024:.1f} KB')

# ─── Quick validation: compare block outputs to PyTorch reference ─────────────
print('\nComparison to PyTorch reference:')
print(f'{"Block":>7}  {"Component":>10}  {"MaxAbsErr":>12}  {"L2RelErr":>12}')
print('-' * 50)
for _l in range(DEPTH):
    _blk_key = f'block{_l}'
    _blk_result = (results['blocks'].get(_blk_key)
                   if 'blocks' in results
                   else results.get('prefill', {}).get(_blk_key))
    if _blk_result is None:
        continue
    for _comp in ('ln1', 'mixing', 'ln2', 'polymlp', 'block'):
        if _comp not in _blk_result or _comp not in pt_refs.get(_l, {}):
            continue
        _fhe_h  = np.array(_blk_result[_comp]['hidden'])
        _pt_h   = pt_refs[_l][_comp]
        # align shapes (generation step has (1,D), pt_refs may be (T,D))
        if _fhe_h.shape != _pt_h.shape:
            continue
        _max_e  = float(np.abs(_fhe_h - _pt_h).max())
        _l2_rel = float(np.linalg.norm(_fhe_h - _pt_h) / (np.linalg.norm(_pt_h) + 1e-12))
        print(f'  {_l:>5}  {_comp:>10}  {_max_e:>12.4e}  {_l2_rel:>12.4e}')

Results saved → /data2/users/gshin22/fhe_output/results/C_20260428_111457.json
File size: 3644.9 KB

Comparison to PyTorch reference:
  Block   Component     MaxAbsErr      L2RelErr
--------------------------------------------------
      0         ln1    3.6796e-03    5.5080e-04
      0      mixing    3.1600e+01    1.5579e-02
      0         ln2    1.6408e-01    1.9316e-02
      0     polymlp    4.8642e+00    6.0729e-03
      0       block    3.2197e+01    1.3888e-02
      1         ln1    2.1086e-01    2.4527e-02
      1      mixing    6.3611e+00    6.6736e-03
      1         ln2    1.8330e-01    2.7932e-02
      1     polymlp    1.3014e+01    1.0975e-02
      1       block    3.1994e+01    1.2222e-02
      2         ln1    2.3653e-01    3.3005e-02
      2      mixing    1.8050e+01    1.5038e-02
      2         ln2    3.3042e-01    4.1019e-02
      2     polymlp    9.9656e+00    1.1793e-02
      2       block    4.2140e+01    1.1390e-02
      3         ln1    4.2309e-01    4.8274e-02